# Final Findings Summary
## BNPL Project ranking assignment

This notebook uncovers the mian discoveries throughtout the analytical section of the Buy Now Pay Later Assignment for project 2 of Applied Data Science. Ultimately, we wish to rank the top 100 merchants out eh the >4000 supplied to us to decide who is the most reliable to onboard.  

## Stage 1: Loading & Cleaning

This is unambiguously boring, however a few cells have been set aside to look into the handling of duplicates and trimming of extreme values, since a slightly alternative path was opted for by rarely dropping larger or smaller values. 

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession, Window, functions as F

# Works when launched from the repository, project2, or notebooks directory.
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if p.name == "project2"), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd() / "project2"
PROCESSED_PATH = PROJECT_ROOT / "data" / "tables" / "processed"
if not PROCESSED_PATH.is_dir():
    raise FileNotFoundError("Set PROJECT_ROOT to your project2 directory.")
RANDOM_STATE = 42
spark = SparkSession.builder.config("spark.driver.memory", "6g").getOrCreate()

import statsmodels.api as sm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer, PowerTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import boxcox

from scipy.stats import norm
AMOUNT_CUTOFF = 8000.0
MAX_PLOT_ROWS = 100_000
RANDOM_STATE = 42

# Shared look for the distribution plots: light blue histogram, thin red KDE line.
HIST_BLUE, KDE_RED = "#9ecae1", "#d62728"


In [14]:
transactions = spark.read.parquet(
    str(PROCESSED_PATH / "transactions_clean")
)

merchants = spark.read.parquet(
    str(PROCESSED_PATH / "merchants_clean")
)

consumers = spark.read.parquet(
    str(PROCESSED_PATH / "consumers")
)

duplicates = spark.read.parquet(
    str(PROCESSED_PATH / "duplicate_order_ids")
)

n_tx = transactions.count()
tx_merchants = transactions.select("merchant_abn").distinct()
tx_users = transactions.select("user_id").distinct()

print("Duplicate order rows:", n_tx - transactions.select("order_id").distinct().count())
print("Transacting merchants with no metadata:",
      tx_merchants.join(merchants, "merchant_abn", "left_anti").count())
print("Registered merchants that never transact:",
      merchants.join(tx_merchants, "merchant_abn", "left_anti").count())
print("Transacting users missing from consumers:",
      tx_users.join(consumers, "user_id", "left_anti").count())
print("Registered consumers that never transact:",
      consumers.join(tx_users, "user_id", "left_anti").count())


Duplicate order rows: 0
Transacting merchants with no metadata: 396
Registered merchants that never transact: 0
Transacting users missing from consumers: 0
Registered consumers that never transact: 475918


## Outlier analysis and distributions

For an idea on the shape of numerical data we now have a look at some of the distributional plots of our (collected) dataset.

In [ ]:
value = F.col("dollar_value").cast("double")
valid = value.isNotNull() & ~F.isnan(value) & (F.abs(value) != float("inf")) & (value >= 0)
cost = transactions.select(value.alias("dollar_value")).filter(valid)
valid_n = cost.count()

# Bound driver-memory use; plots use a seeded Spark sample, capped at MAX_PLOT_ROWS.
fraction = min(1.0, MAX_PLOT_ROWS / valid_n)
plot_values = (cost.sample(False, fraction, seed=RANDOM_STATE).limit(MAX_PLOT_ROWS)
               .toPandas()["dollar_value"].to_numpy())

log_cost = cost.withColumn("log_cost", F.log1p("dollar_value"))
log_values = np.log1p(plot_values)
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(x=log_values, bins=120, stat="density", color=HIST_BLUE, edgecolor="none",
             alpha=0.7, ax=ax)
sns.kdeplot(x=log_values, color=KDE_RED, linewidth=1, ax=ax)
ticks = np.arange(0, int(np.ceil(log_values.max())) + 1)
ax.set_xticks(ticks, [f"{np.expm1(t):,.0f}" for t in ticks], rotation=45)
ax.set(xlabel="Transaction amount ($; log1p spacing)", ylabel="Density",
       title="Log-transformed amount distribution")
sns.despine(ax=ax)
fig.tight_layout()
plt.show()


## Market segments

Merchants were grouped with k-means (k = 4) on six features from `merchant_features`, each converted to a percentile rank and standardised before clustering (`scripts/segment_behaviour.py`).

| Dimension | Column | In the v2 ranking? |
|---|---|---|
| Commercial value | `estimated_bnpl_revenue` | yes, 70% |
| Customer reach | `customer_penetration` (unique customers ÷ 24,081) | no; `unique_customers` was dropped from v2 as collinear with revenue |
| Engagement | `transactions_per_customer` | yes, 4% |
| Growth | `log_value_growth` | yes, 8% |
| Historical activity | `active_month_ratio` | yes, 2% |
| Volatility | `value_cv` | yes, 3% (sign flipped, as "stability") |


In [ ]:
# Estimated BNPL revenue per merchant, by behaviour segment (scripts/segment_behaviour.py).
COL = "estimated_bnpl_revenue"
NAMES = {1: "Established", 3: "Rising", 2: "Fading", 0: "Intermittent"}

segments = pd.read_parquet(PROCESSED_PATH / "merchant_behaviour_segments")[["merchant_abn", "segment_id"]]
ranking = pd.read_parquet(PROCESSED_PATH / "merchant_ranking_v2.parquet")
seg_df = ranking.merge(segments, on="merchant_abn", how="inner")

# Revenue spans several orders of magnitude, so bin and smooth on log10 dollars.
log_rev = np.log10(seg_df[COL])
bins = np.linspace(log_rev.min(), log_rev.max(), 81)

fig, axes = plt.subplots(4, 1, figsize=(8, 8), sharex=True)
for ax, seg_id in zip(axes, [1, 3, 2, 0]):
    block = log_rev[seg_df["segment_id"] == seg_id]
    sns.histplot(x=block, bins=bins, stat="density", color=HIST_BLUE, edgecolor="none",
                 alpha=0.7, ax=ax)
    sns.kdeplot(x=block, color=KDE_RED, linewidth=1, ax=ax)
    ax.set_title(f"{NAMES[seg_id]}  (n = {len(block):,}, median ${10 ** block.median():,.0f})",
                 loc="left", fontsize=10)
    ax.set_ylabel("Density")
    sns.despine(ax=ax)
ticks = np.arange(np.floor(log_rev.min()), np.ceil(log_rev.max()) + 1)
axes[-1].set_xticks(ticks, [f"${10 ** t:,.0f}" for t in ticks])
axes[-1].set_xlabel("Estimated BNPL revenue over the full window (log scale)")
fig.tight_layout()
plt.show()


## Ranking model

Each of the 4,026 merchants with a take rate gets a score from 0 to 100 (`merchant_ranking_v2.ipynb`):

1. **Every feature becomes a percentile rank.** Revenue is in dollars and repeat rate is a fraction, so raw values can't be added; ranks can, and they are not distorted by the few very large merchants.
2. **Thin evidence is pulled towards the middle.** Revenue and growth ranks are shrunk towards 50 by `n / (n + 300)`, where `n` is the merchant's transaction count. A merchant with 40 transactions keeps 12% of its own rank; one with 3,000 keeps 91%.
3. **Missing features score 50**, so a merchant is neither rewarded nor punished for a gap.
4. **The score is a weighted sum** of those ranks. The top 100 by score is the shortlist.

The first version (v1) spread the weight across ten features. A time-split test showed past revenue carried almost all of the predictive power, so v2 moves most of the weight onto it and keeps the other features small.


In [ ]:
# v1 weights as set in merchant_ranking_baseline.ipynb; v2 weights as saved by merchant_ranking_v2.ipynb.
WEIGHTS_V1 = {"estimated_bnpl_revenue": 0.30, "log_value_growth": 0.20, "transactions_per_customer": 0.10,
              "stability": 0.10, "unique_customers": 0.05, "active_month_ratio": 0.05, "repeat_rate": 0.05,
              "customer_income": 0.05, "customer_irsad": 0.05, "customer_spread": 0.05}
weights_v2 = pd.read_csv(PROCESSED_PATH / "weights_v2.csv", index_col=0)["weight"]

weights = pd.DataFrame({"v1": pd.Series(WEIGHTS_V1), "v2 (shipped)": weights_v2}).fillna(0) * 100
weights = weights.sort_values("v2 (shipped)", ascending=False)
weights_long = (weights.rename_axis("feature").reset_index()
                .melt("feature", var_name="version", value_name="weight"))

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(data=weights_long, y="feature", x="weight", hue="version",
            palette=["#b0b0b0", "#4C78A8"], ax=ax)
for bars in ax.containers:
    ax.bar_label(bars, fmt="%.0f%%", padding=3, fontsize=8)
ax.set(xlabel="Weight in the score (%)", ylabel="",
       title="Ranking weights before and after the time-split validation")
ax.legend(frameon=False, loc="lower right", title=None)
sns.despine(ax=ax)
fig.tight_layout()
plt.show()

### Does it predict?

The ranking was built on 28 Feb 2021 – 28 Feb 2022 only, then checked against what merchants actually earned the firm over 1 Mar – 31 Aug 2022. Capture rate is the revenue of the predicted top 100 as a share of the best possible top 100 (\$10.55M). Figures are from `merchant_ranking_v2.ipynb` §6.

| Ranking | Spearman with realised revenue | Capture rate | Picks in the realised top 100 |
|---|---|---|---|
| v1: ten features | 0.740 | 74.3% | 57 |
| **v2: shipped** | **0.905** | **85.4%** | **73** |
| Past revenue alone | 0.977 | 99.9% | 99 |

v2 is a clear improvement on v1, and it still loses to ranking on past revenue alone: about \$1.54M of capture over six months. That is the price of the 30% of the score spent on growth, engagement, concentration and customer base, which this six-month window cannot reward.


In [ ]:
# The top of the shipped list (before the fraud penalty).
RANK_COLS = ["rank_overall", "name", "score", "take_rate", "estimated_bnpl_revenue",
             "transaction_count", "unique_customers", "log_value_growth"]
ranking.nsmallest(10, "rank_overall")[RANK_COLS].round(2)

## Fraud: models to understand the scores, arithmetic to price them

The supplied fraud scores cover under 1% of transactions (34,765 consumer user-days and 114 merchant-days), and none is below 8. They are a pre-screened sample, so we did not impute the missing ones. The fraud work has two separate parts:

**1. Models were used to understand why a score is given.** XGBoost classifiers were trained to predict whether a scored day is flagged (score ≥ 30). They are diagnostic: they show what the scores respond to. They are **not** an input to the ranking.

- Consumer scores follow the **size of the transaction**.
- Merchant scores follow how large the day is **relative to that merchant's usual day**.

The cell below loads the trained models saved by `fraud_classifier.ipynb` and `merchant_fraud_penalty.ipynb` (in `data/tables/models/`) and plots their predictions on held-out data against that single driver.

**2. The penalty uses the supplied scores directly, with no model.** For every transaction in the scoring window:

- $p_\text{consumer}$ = consumer score ÷ 100, split across that user's purchases that day in proportion to dollar value
- $p_\text{merchant}$ = merchant score ÷ 100
- $P(\text{fraud}) = 1 - (1 - p_\text{merchant})(1 - p_\text{consumer})$
- expected loss $= P(\text{fraud}) \times$ transaction value

Scores below 30 count as zero. Summing per merchant gives a loss rate, which is subtracted from the merchant's BNPL revenue before it is ranked. So the penalty depends on how well calibrated the supplied scores are, not on the accuracy of any model we trained.


In [ ]:
from sklearn.metrics import average_precision_score
from xgboost import XGBClassifier

# Trained models and their evaluation data are exported by the last cell of the
# classifier section in fraud_classifier.ipynb and merchant_fraud_penalty.ipynb.
MODEL_DIR = PROJECT_ROOT / "data" / "tables" / "models"
FLAG_CUTOFF = 30   # a score at or above this is treated as a fraud flag


def load_model(name):
    path = MODEL_DIR / f"{name}.json"
    if not path.exists():
        raise FileNotFoundError(f"{path} is missing - run the notebook that trains and saves it first.")
    model = XGBClassifier()
    model.load_model(path)
    return model


def top_features(model, n=4):
    gain = pd.Series(model.get_booster().get_score(importance_type="gain"))
    return ", ".join(f"{name} ({share:.0%})" for name, share in (gain / gain.sum()).nlargest(n).items())


# --- Consumer side: the saved XGBoost model applied to users it never saw in training.
consumer_model = load_model(f"consumer_fraud_ge{FLAG_CUTOFF}")
consumer_test = pd.read_parquet(MODEL_DIR / "consumer_fraud_test.parquet")
consumer_test["flagged"] = (consumer_test["fraud_probability"] >= FLAG_CUTOFF).astype(int)
consumer_test["p_flagged"] = consumer_model.predict_proba(
    consumer_test[consumer_model.get_booster().feature_names])[:, 1]
# Days with one transaction, where the score belongs to exactly that transaction.
consumer_single = consumer_test[consumer_test["day_txn_count"] == 1]

# --- Merchant side: 114 scored days. The saved model was fitted on all of them, so the
# predictions plotted are the out-of-fold ones (each merchant predicted by a model
# trained on the other 60), exported alongside it.
merchant_model = load_model(f"merchant_fraud_ge{FLAG_CUTOFF}")
merchant_days = pd.read_parquet(MODEL_DIR / "merchant_fraud_days.parquet")
merchant_days["flagged"] = (merchant_days["fraud_probability"] >= FLAG_CUTOFF).astype(int)
merchant_days["p_flagged"] = merchant_days[f"p_ge_{FLAG_CUTOFF}_oof"]

for label, frame, model in [("consumer", consumer_single, consumer_model),
                            ("merchant", merchant_days, merchant_model)]:
    print(f"{label}: {len(frame):,} held-out scored days, {frame['flagged'].sum()} flagged | "
          f"PR-AUC {average_precision_score(frame['flagged'], frame['p_flagged']):.2f} "
          f"(random = {frame['flagged'].mean():.2f})")
    print(f"  model leans on: {top_features(model)}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
panels = [
    (axes[0], consumer_single, "dollar_value", 10, "Transaction value ($, log scale)",
     "Consumer: flagged when the purchase is large"),
    (axes[1], merchant_days, "revenue_ratio", 30, "Day's revenue ÷ merchant's median day (log scale)",
     "Merchant: flagged when the day is a spike"),
]
for ax, frame, x, size, xlabel, title in panels:
    for flagged, colour, name in [(0, HIST_BLUE, f"score < {FLAG_CUTOFF}"), (1, KDE_RED, f"score ≥ {FLAG_CUTOFF}")]:
        part = frame[frame["flagged"] == flagged]
        ax.scatter(part[x], part["p_flagged"], s=size, color=colour, alpha=0.6, edgecolor="none", label=name)
    ax.set_xscale("log")
    ax.set(xlabel=xlabel, title=title)
    sns.despine(ax=ax)
axes[0].set_ylabel(f"XGBoost predicted P(score ≥ {FLAG_CUTOFF}), held out")
axes[0].legend(frameon=False, loc="upper left", title="Actual score")
fig.tight_layout()
plt.show()

## Fraud penalty: what it changes in the ranking

Compares the v2 ranking (no fraud) with the fraud-adjusted ranking from `merchant_fraud_penalty.ipynb`. `score_change` is the fraud-adjusted score minus the v2 score, so the most negative values are the merchants the penalty hit hardest.


In [ ]:
fraud = pd.read_parquet(PROCESSED_PATH / "merchant_ranking_v2_fraud_adjusted.parquet")
fraud["score_change"] = fraud["score_adjusted"] - fraud["score"]
penalised = fraud["expected_fraud_loss"] > 0

print(f"{penalised.sum():,} of {len(fraud):,} merchants have some expected fraud loss; "
      f"{(fraud['score_change'] < 0).sum():,} end up with a lower score")

# The merchants the penalty moved furthest down.
FRAUD_COLS = ["name", "rank_overall", "rank_adjusted", "score", "score_adjusted", "score_change",
              "estimated_bnpl_revenue", "expected_fraud_loss", "loss_to_revenue",
              "avg_transaction_value", "transaction_count", "risk_tier"]
fraud.nsmallest(15, "score_change")[FRAUD_COLS].round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.scatterplot(data=fraud[~penalised], x="avg_transaction_value", y="score_change", s=8,
                color="#b0b0b0", edgecolor="none", alpha=0.5, label="no expected loss", ax=ax)
sns.scatterplot(data=fraud[penalised], x="avg_transaction_value", y="score_change", s=14,
                color=KDE_RED, edgecolor="none", alpha=0.7, label="has expected fraud loss", ax=ax)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xscale("log")
ax.set(xlabel="Merchant's average transaction value ($, log scale)",
       ylabel="Score change (fraud-adjusted - v2)",
       title="Fraud penalty by merchant ticket size")
ax.legend(frameon=False, loc="lower left")
sns.despine(ax=ax)
fig.tight_layout()
plt.show()